In [11]:
!pip install icrawler
!pip install split-folders

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
from icrawler.builtin import BingImageCrawler
import os

keywords = [
    "Karnak Temple Luxor Egypt",
    "Luxor Temple Egypt",
    "Valley of the Kings Luxor",
    "Hatshepsut Temple Luxor",
    "Colossi of Memnon Luxor"
]

os.makedirs("dataset_raw", exist_ok=True)

for keyword in keywords:
    folder_name = keyword.replace(' ', '_')
    path = f"dataset_raw/{folder_name}"
    os.makedirs(path, exist_ok=True)

    crawler = BingImageCrawler(storage={"root_dir": path})
    crawler.crawl(keyword=keyword, max_num=60)

    print(f"Finished {keyword}")


✅ Finished Karnak Temple Luxor Egypt


ERROR:downloader:Response status code 403, file https://www.re-thinkingthefuture.com/wp-content/uploads/2022/10/A8320-Luxor-Temple-Egypt-Image-4-1024x681.jpg
ERROR:downloader:Response status code 403, file https://www.re-thinkingthefuture.com/wp-content/uploads/2022/10/A8320-Luxor-Temple-Egypt-Image-6-1536x384.jpg


✅ Finished Luxor Temple Egypt


ERROR:downloader:Response status code 400, file https://media.istockphoto.com/id/1150128932/photo/valley-of-kings-on-west-bank-of-nile-river-in-luxor-egypt.jpg
ERROR:downloader:Response status code 403, file https://deih43ym53wif.cloudfront.net/tomb-ramesses-vi-valley-kings-luxor-shutterstock_2282005833.jpg


✅ Finished Valley of the Kings Luxor
✅ Finished Hatshepsut Temple Luxor
✅ Finished Colossi of Memnon Luxor


In [12]:
import splitfolders
splitfolders.ratio("dataset_raw", output="dataset", seed=42, ratio=(0.8,0.2))

Copying files: 300 files [00:00, 2739.02 files/s]


In [13]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset/val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 240 files belonging to 5 classes.
Found 60 files belonging to 5 classes.


In [14]:
from tensorflow.keras import layers

# Augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2)
])
train_ds = train_ds.map(lambda x,y: (data_augmentation(x, training=True), y))

# Normalization
normalization_layer = layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x,y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x,y: (normalization_layer(x), y))


In [15]:
from tensorflow.keras import models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers

base_model = MobileNetV2(input_shape=(224,224,3), include_top=False, weights='imagenet')
base_model.trainable = False  # Freeze base

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(5, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [20]:
import os
from PIL import Image

dataset_path = "dataset_raw"

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        file_path = os.path.join(root, file)
        try:
            img = Image.open(file_path)
            img.verify()
        except:
            print("Deleting:", file_path)
            os.remove(file_path)

In [24]:
import os
import imghdr

dataset_path = "dataset_raw"

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        file_path = os.path.join(root, file)
        try:
            if imghdr.what(file_path) not in ["jpeg", "png", "bmp", "gif"]:
                print("Deleting invalid file:", file_path)
                os.remove(file_path)
        except:
            os.remove(file_path)

Deleting invalid file: dataset_raw/Valley_of_the_Kings_Luxor/000043.jpg
Deleting invalid file: dataset_raw/Valley_of_the_Kings_Luxor/000044.jpg
Deleting invalid file: dataset_raw/Valley_of_the_Kings_Luxor/000045.jpg
Deleting invalid file: dataset_raw/Luxor_Temple_Egypt/000006.jpg


/tmp/ipython-input-2979390552.py:2: DeprecationWarning: 'imghdr' is deprecated and slated for removal in Python 3.13
  import imghdr


In [34]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

In [36]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False
model.fit(train_ds, validation_data=val_ds, epochs=10)

Epoch 1/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 22s 3s/step - accuracy: 0.8015 - loss: 0.6260 - val_accuracy: 0.6271 - val_loss: 1.0212
Epoch 2/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step - accuracy: 0.8895 - loss: 0.5079 - val_accuracy: 0.6271 - val_loss: 1.0227
Epoch 3/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 26s 3s/step - accuracy: 0.8385 - loss: 0.5871 - val_accuracy: 0.6271 - val_loss: 1.0254
Epoch 4/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 24s 3s/step - accuracy: 0.9144 - loss: 0.4737 - val_accuracy: 0.6102 - val_loss: 1.0271
Epoch 5/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 40s 3s/step - accuracy: 0.9450 - loss: 0.4557 - val_accuracy: 0.6271 - val_loss: 1.0306
Epoch 6/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 40s 3s/step - accuracy: 0.8837 - loss: 0.4933 - val_accuracy: 0.6271 - val_loss: 1.0343
Epoch 7/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step - accuracy: 0.8981 - loss: 0.4125 - val_accuracy: 0.6271 - val_loss: 1.0393
Epoch 8/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 42s 3s/step - accuracy: 0.9256 - loss: 0.4291 - val_accuracy: 0.6102 - val_loss: 1.0389
Epoch 9/

In [41]:
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(train_ds, validation_data=val_ds, epochs=8)


Epoch 1/8
8/8 ━━━━━━━━━━━━━━━━━━━━ 42s 3s/step - accuracy: 0.9265 - loss: 0.4066 - val_accuracy: 0.5932 - val_loss: 1.0417
Epoch 2/8
8/8 ━━━━━━━━━━━━━━━━━━━━ 42s 3s/step - accuracy: 0.9571 - loss: 0.2961 - val_accuracy: 0.6102 - val_loss: 1.0268
Epoch 3/8
8/8 ━━━━━━━━━━━━━━━━━━━━ 24s 3s/step - accuracy: 0.9326 - loss: 0.3490 - val_accuracy: 0.6102 - val_loss: 1.0209
Epoch 4/8
8/8 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step - accuracy: 0.9683 - loss: 0.2739 - val_accuracy: 0.6102 - val_loss: 1.0169
Epoch 5/8
8/8 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step - accuracy: 0.9565 - loss: 0.2828 - val_accuracy: 0.6102 - val_loss: 1.0160
Epoch 6/8
8/8 ━━━━━━━━━━━━━━━━━━━━ 25s 3s/step - accuracy: 0.9760 - loss: 0.2320 - val_accuracy: 0.6102 - val_loss: 1.0258
Epoch 7/8
8/8 ━━━━━━━━━━━━━━━━━━━━ 24s 3s/step - accuracy: 0.9824 - loss: 0.2069 - val_accuracy: 0.6102 - val_loss: 1.0465
Epoch 8/8
8/8 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step - accuracy: 0.9872 - loss: 0.1748 - val_accuracy: 0.6102 - val_loss: 1.0646


In [42]:
os.makedirs("saved_model", exist_ok=True)
model.save("saved_model/luxor_landmark_model.keras")
print("Model saved ")

Model saved 
